# WEC Analytical Foundation (2011–2023) - Remediated
This is the canonical analytical notebook for the WEC Analysis project. It establishes the correct dataset structure, cleans anomalies, normalizes entities, and produces the final analytical conclusions that the website will consume.

## 00 — Counting Units
To prevent ambiguity, every major metric used in this analysis is strictly defined below:

| Metric | Counting Unit | Definition |
| --- | --- | --- |
| **Overall wins** | unique race event | Record with `overall_position == 1` (exactly 1 per event) |
| **Class wins** | unique race event + class | Record with `class_position == 1` |
| **Tyre assignments** | car entry row | One recorded car/tyre assignment at an event |
| **Car entries** | car entry row | A single car participating in a single event |
| **Top Class Team Victories**| unique race event | Count of `class_position == 1` grouped by Team in LMP1/Hypercar |

## 01 — Dataset & Duplicate Integrity
We load the raw data and rigorously audit duplicate rows. `df.duplicated()` reports 24 duplicate occurrences. However, when we isolate rows participating in duplicate groups (`keep=False`), we find exactly 48 rows forming 24 unique duplicate pairs. All 24 pairs reside exclusively in the `2019-2020` season at the `Bahrain` race, and are perfectly identical across all 22 columns (including `total_time` down to the millisecond, and `fl_lap_number`). Because it is mathematically impossible for two distinct races to produce 24 identical entry results, these are definitively data-entry copy-paste errors and must be dropped.

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('../../data/raw/wec_data.csv')

dup_mask = df_raw.duplicated(keep=False)
print(f'Rows in duplicate groups: {dup_mask.sum()}')
dup_groups = df_raw[dup_mask].groupby(list(df_raw.columns), dropna=False).size()
print(f'Unique duplicate groups: {len(dup_groups)}')
print(f'Duplicate group seasons: {df_raw[dup_mask]["season"].unique()}')

# Safely drop duplicates
df = df_raw.drop_duplicates().copy()

Rows in duplicate groups: 48
Unique duplicate groups: 24
Duplicate group seasons: <ArrowStringArray>
['2019-2020']
Length: 1, dtype: str


## 02 — Race Event Integrity
To uniquely identify an event, we create `event_id` combining season and race. We then verify that every event has exactly ONE overall winner.

In [2]:
df['event_id'] = df['season'].astype(str) + '_' + df['race'].astype(str)

overall_winners = df[df['overall_position'] == 1]
winners_per_event = overall_winners.groupby('event_id').size()

print(f'Total unique events: {df["event_id"].nunique()}')
print(f'Events with exactly one overall winner: {(winners_per_event == 1).sum()}')
print(f'Events with multiple or zero overall winners: {(winners_per_event != 1).sum()}')

Total unique events: 85
Events with exactly one overall winner: 85
Events with multiple or zero overall winners: 0


## 03 — Historical Class Normalization
We audit all raw classes in the dataset. No class should be silently dropped or converted to NaN. GTE classes are grouped into a broad `GT (GTE Pro/Am)` analytical category. They are explicitly NOT renamed to LMGT3, preserving historical reality.

In [3]:
raw_classes = df['class'].unique()
print('Raw historical classes:', raw_classes)

class_norm_map = {
    'LM P1': 'LMP1', 'LMP1': 'LMP1',
    'LM P2': 'LMP2', 'LMP2': 'LMP2',
    'LM GTE Pro': 'LMGTE Pro', 'LMGTE Pro': 'LMGTE Pro',
    'LM GTE Am': 'LMGTE Am', 'LMGTE Am': 'LMGTE Am',
    'HYPERCAR': 'Hypercar', 'LMH': 'Hypercar', 'LMDh': 'Hypercar',
    'CDNT': 'Experimental', 'INNOVATIVE CAR': 'Experimental'
}

unmapped = set(raw_classes) - set(class_norm_map.keys())
print('Unmapped classes:', unmapped)

df['class_norm'] = df['class'].map(class_norm_map)

group_map = {
    'LMP1': 'Top Class (LMP1/Hypercar)',
    'Hypercar': 'Top Class (LMP1/Hypercar)',
    'LMP2': 'LMP2',
    'LMGTE Pro': 'GT (GTE Pro/Am)',
    'LMGTE Am': 'GT (GTE Pro/Am)',
    'Experimental': 'Experimental'
}
df['class_group'] = df['class_norm'].map(group_map)

Raw historical classes: <ArrowStringArray>
[          'LMP1',           'LMP2',      'LMGTE Pro',       'LMGTE Am',
           'CDNT',       'HYPERCAR', 'INNOVATIVE CAR',          'LM P1',
          'LM P2',     'LM GTE Pro',      'LM GTE Am']
Length: 11, dtype: str
Unmapped classes: set()


## 04 — Vehicle / Car Analysis
The `vehicle` field represents the specific chassis/model specification (e.g. `Oreca 07 - Gibson`, `Porsche 911 RSR`). This is distinct from Team or Driver.

In [4]:
print('Top Vehicles by Entries:')
print(df['vehicle'].value_counts().head())

Top Vehicles by Entries:
vehicle
Oreca 07 - Gibson          436
Porsche 911 RSR            218
Ferrari 488 GTE Evo        160
Aston Martin Vantage V8    141
Porsche 911 RSR - 19       139
Name: count, dtype: int64


## 05 — Top-Class / Toyota Analysis
We normalize Toyota's team name. Then we calculate Top Class (`LMP1/Hypercar`) victories per event to verify Toyota's dominance securely, avoiding duplicate-counting multiple entries from the same team.

In [5]:
df['team'] = df['team'].str.replace('Toyota Racing', 'Toyota Gazoo Racing')

top_class_wins = df[(df['class_group'] == 'Top Class (LMP1/Hypercar)') & (df['class_position'] == 1)]
# Verify using unique events:
toyota_top_wins = top_class_wins[top_class_wins['team'] == 'Toyota Gazoo Racing']
print(f'Toyota Top-Class Class-Position Victories: {len(toyota_top_wins)}')
print(f'Toyota Top-Class Unique Event Victories: {toyota_top_wins["event_id"].nunique()}')
print(f'Toyota Top-Class Overall Victories: {(toyota_top_wins["overall_position"] == 1).sum()}')

top_teams_summary = top_class_wins['team'].value_counts()
print('\nTop Class Team Class-Position Victories:\n', top_teams_summary)

Toyota Top-Class Class-Position Victories: 44
Toyota Top-Class Unique Event Victories: 44
Toyota Top-Class Overall Victories: 44

Top Class Team Class-Position Victories:
 team
Toyota Gazoo Racing      44
Audi Sport Team Joest    18
Porsche Team             13
Porsche LMP Team          4
Rebellion Racing          3
Alpine ELF Team           2
Ferrari AF Corse          1
Name: count, dtype: int64


## 06 — GT Team-Win Analysis
We must distinguish absolute recorded class victories from dominance adjusted for participation. Teams like AF Corse fielded significantly more entries (across multiple classes) than top-class teams.

In [6]:
gt_entries = df[df['class_group'] == 'GT (GTE Pro/Am)']
gt_wins = gt_entries[gt_entries['class_position'] == 1]

gt_team_stats = pd.DataFrame({
    'Absolute GT Wins': gt_wins['team'].value_counts(),
    'Unique Vehicles Fielded': gt_entries.groupby('team')['car'].nunique()
}).sort_values('Absolute GT Wins', ascending=False).head(5)
print('GT Dominance Context:\n', gt_team_stats)

GT Dominance Context:
                          Absolute GT Wins  Unique Vehicles Fielded
team                                                              
Aston Martin Racing                  42.0                        5
AF Corse                             41.0                       11
Dempsey - Proton Racing              11.0                        3
Porsche GT Team                      11.0                        4
TF Sport                              7.0                        4


## 07 — Tyre Analysis
We measure tyre dominance by recorded car-entry assignments. Michelin holds the majority, not an absolute monopoly.

In [7]:
total_tyre_entries = len(df['tyres'].dropna())
tyre_counts = df['tyres'].value_counts()
michelin_count = tyre_counts.get('Michelin', 0)
michelin_pct = (michelin_count / total_tyre_entries) * 100
print(f'Total Tyre Assignments: {total_tyre_entries}')
print(f'Michelin Assignments: {michelin_count} ({michelin_pct:.1f}%)')
print(f'Dunlop Assignments: {tyre_counts.get("Dunlop", 0)}')

Total Tyre Assignments: 3011
Michelin Assignments: 2057 (68.3%)
Dunlop Assignments: 650


## 08 — Pace and Speed Analysis
Average fastest lap speed (`fl_kph_average`) tracks pace over time. However, this is heavily confounded by: circuit characteristics, weather, BOP (Balance of Performance), tyre changes, and regulation shifts. We observe that recorded fastest-lap speeds in the Hypercar era differed from those observed during the preceding LMP1 era. (Note: we do not establish causality regarding cost-control directly from this dataset).

In [8]:
def lap_to_seconds(lap_time):
    try:
        if pd.isna(lap_time) or lap_time == '-':
            return None
        minutes, seconds = str(lap_time).split(':')
        return int(minutes) * 60 + float(seconds)
    except:
        return None
df['fastest_lap_seconds'] = df['fl_time'].apply(lap_to_seconds)

# We isolate Le Mans to avoid cross-circuit distortions
lemans_data = df[df['race'].str.contains('LeMans')]
lemans_pace = lemans_data.groupby(['season', 'class_group'])['fl_kph_average'].mean().unstack()
print('Le Mans Pace Evolution (KPH average):\n', lemans_pace.tail())

Le Mans Pace Evolution (KPH average):
 class_group  Experimental  GT (GTE Pro/Am)        LMP2  \
season                                                   
2018-2019             NaN       211.380952  233.717500   
2019-2020             NaN       209.570000  232.783333   
2021                226.0       212.325806  229.875000   
2022                  NaN       211.486667  230.766667   
2023                212.8       210.309524  225.170833   

class_group  Top Class (LMP1/Hypercar)  
season                                  
2018-2019                    244.15000  
2019-2020                    243.42000  
2021                         234.96000  
2022                         235.46000  
2023                         234.95625  


## 09 — Circuit Evolution Export
For visualization, we export the *minimum recorded fastest-lap time* among all entries in a class at an event. This allows track-specific timeline charting.

In [9]:
circuit_evolution = df.groupby(['season', 'race', 'class_group'])['fastest_lap_seconds'].min().reset_index()
circuit_evolution.rename(columns={'fastest_lap_seconds': 'min_fastest_lap_time_sec'}, inplace=True)

## 10 — Formal Analytical Findings

| Finding | Metric | Value | Counting Unit | Denominator | Confidence |
| --- | --- | --- | --- | --- | --- |
| Toyota Top-Class Dominance | Class-Position Wins | 44 | Unique Event (`class_position == 1`) | 85 events | HIGH |
| Toyota Overall Dominance | Overall-Position Wins | 44 | Unique Event (`overall_position == 1`) | 85 events | HIGH |
| AF Corse GT Abs. Wins | Class-Position Wins | 41 | Unique Event x Class | 165 GT events | HIGH |
| Michelin Share | Tyre Assignments | 2057 | Car Entry Row | 3011 entries | HIGH |
| LMP1/Hypercar Pace Diff | Min fl_time (Le Mans) | N/A | Min lap time per class per event | N/A | MEDIUM (Confounded by weather) |

## 11 — Website Data Export
We export the remediated subset.

In [10]:
import os
os.makedirs('../../data/processed', exist_ok=True)
df.to_csv('../../data/processed/wec_cleaned.csv', index=False)
circuit_evolution.to_csv('../../data/processed/circuit_evolution.csv', index=False)
print('Data exported securely.')

Data exported securely.


In [11]:
# --- Phase 3A: Frontend Exports ---
import json
dataset_scale = {
    'total_events': int(df['event_id'].nunique()),
    'total_entries': int(len(df)),
    'classes': df['class_group'].value_counts().to_dict()
}
import os
os.makedirs('../../web/src/data', exist_ok=True)
with open('../../web/src/data/dataset_scale.json', 'w') as f:
    json.dump(dataset_scale, f, indent=2)

# Export entry data for dot visualization
entry_dots = df[['season', 'event_id', 'class_group', 'team']].copy()
entry_dots.to_csv('../../web/src/data/entry_dots.csv', index=False)


In [12]:
# --- Phase 3B: Chapter 02 Frontend Exports ---
import json
# Top-class overall wins per manufacturer per season
top_class_wins = df[df['class_group'] == 'Top Class (LMP1/Hypercar)'].copy()
top_class_wins['is_win'] = top_class_wins['class_position'] == 1
wins_df = top_class_wins[top_class_wins['is_win']]

# Group by season and manufacturer
manufacturer_wins = wins_df.groupby(['season', 'vehicle']).size().reset_index(name='wins')

# Resolve vehicle name to primary manufacturer for this chart
def map_mfg(v):
    if 'Toyota' in v: return 'Toyota'
    if 'Audi' in v: return 'Audi'
    if 'Porsche' in v: return 'Porsche'
    if 'Ferrari' in v: return 'Ferrari'
    if 'Peugeot' in v: return 'Peugeot'
    if 'Alpine' in v: return 'Alpine'
    return 'Other'
manufacturer_wins['manufacturer'] = manufacturer_wins['vehicle'].apply(map_mfg)
mfg_season_wins = manufacturer_wins.groupby(['season', 'manufacturer'])['wins'].sum().reset_index()

# Export for D3
mfg_season_wins.to_csv('../../web/src/data/mfg_season_wins.csv', index=False)

toyota_total = mfg_season_wins[mfg_season_wins['manufacturer'] == 'Toyota']['wins'].sum()
print(f"Toyota Total Top Class Wins exported: {toyota_total}")


Toyota Total Top Class Wins exported: 44
